# Fig. 3 · Empirical alignment and the d-EPR null

Panel a compares empirical and null scores across Finland and four urban areas. It includes residential H3 L9 cells with at least three users. Panel b uses
the Fig. 2 travel time groups for cells with both alignment scores available.
Panels c–e reproduce the saved destination attribute summaries by urban region.

In [ ]:
from pathlib import Path
import sys

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullLocator
import numpy as np
import pandas as pd
import seaborn as sns

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'figure_style.py').is_file())
sys.path.insert(0, str(root / 'src'))

from figure_style import add_panel_label, configure_style, export_figure

DATA_DIR = root / 'data'
configure_style('final')

SERIES = {
    'empirical': dict(label='Empirical', colour='#1B8B8B', marker='o'),
    'null': dict(label='d-EPR null', colour='#7B5EA7', marker='s'),
}


## a. Regional alignment distributions

The density curves use the same Gaussian KDE as Fig. 1c, with Scott bandwidth scaled by 0.8 and support clipped to [0, 1].

In [ ]:
empirical = pd.read_parquet(DATA_DIR / 'aggregated' / 'empirical_qk_h3.parquet')
null = pd.read_parquet(DATA_DIR / 'aggregated' / 'null_qk_h3.parquet')
regional = (empirical.merge(null, on='home_gid9', validate='one_to_one')
            .rename(columns={'qk_empirical': 'empirical', 'qk_null': 'null'}))
regional = regional.loc[regional.n_users.ge(3)]
regions = {
    "Finland": regional,
    "Helsinki region": regional.loc[regional.is_helsinki_region],
    **{city: regional.loc[regional.name.eq(city)] for city in ("Oulu", "Tampere", "Turku")},
}

with plt.rc_context({"font.size": 10}):
    fig, ax = plt.subplots(figsize=(110 / 25.4, 75 / 25.4), layout="constrained")


    kde_fig, kde_ax = plt.subplots()
    densities = {}
    for region, frame in regions.items():
        for key in SERIES:
            sns.kdeplot(x=frame[key], bw_adjust=0.8, clip=(0, 1), ax=kde_ax)
            line = kde_ax.lines[-1]
            densities[region, key] = line.get_data()
            line.remove()
    plt.close(kde_fig)
    peak = max(density.max() for _, density in densities.values())
    half_width = .94 / 2

    for pos, (region, frame) in enumerate(regions.items()):
        values = frame[list(SERIES)]
        ax.vlines(pos, values.min().min(), values.max().max(), color="#4d4d4d", lw=.7, zorder=3.5)
        for direction, (key, style) in zip((-1, 1), SERIES.items()):
            y, density = densities[region, key]
            edge = pos + direction * half_width * density / peak
            colour = style["colour"]
            label = style["label"] if pos == 0 else "_nolegend_"
            ax.fill_betweenx(y, pos, edge, color=colour, alpha=.29, linewidth=0, label=label)
            ax.plot(edge, y, color=colour, linewidth=.9, linestyle="-", zorder=3)
            q25, median, q75 = frame[key].quantile([.25, .50, .75])
            ax.errorbar(
                pos + direction * half_width * .23, median,
                yerr=[[median - q25], [q75 - median]], fmt=style["marker"],
                color=colour, markerfacecolor="white", markersize=2.8,
                elinewidth=.9, capsize=2.5, zorder=4,
            )

    ax.tick_params(labelsize=9)
    ax.grid(False)
    ax.set_ylabel(r"$q_K$")
    ax.set_xticks(range(len(regions)), list(regions))
    ax.set_xlim(-.52, len(regions) - .48)
    ax.set_yticks([0, .2, .4, .6, .8, 1])
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=2, fontsize=9, loc="lower center", bbox_to_anchor=(.5, 1.02))

add_panel_label(ax, "a")
export_figure(fig, "fig3/fig3a-empirical-null")

## b. Alignment by accessibility

In [ ]:


TIME_BINS = [str(edge) for edge in range(5, 40, 5)] + ['40+']


def proximity_groups(cells):
    """Attach the common T_prox groups used in Figures 2 and 3."""
    result = cells.copy()
    result['T_prox_bin'] = pd.cut(
        result.T_prox, [*range(0, 36, 5), np.inf], labels=TIME_BINS, right=False,
    )
    return result


def summarise_proximity(cells, metrics):
    """Calculate group sizes, mean T_prox and each metric's mean and quartiles."""
    grouped = cells.groupby('T_prox_bin', observed=True)
    summary = grouped.agg(n=('T_prox', 'size'), x=('T_prox', 'mean'))
    for metric in metrics:
        summary[metric + '_mean'] = grouped[metric].mean()
        summary[metric + '_q25'] = grouped[metric].quantile(.25)
        summary[metric + '_q75'] = grouped[metric].quantile(.75)
    return summary.reset_index()

time = pd.read_parquet(DATA_DIR / 'aggregated' / 'travel_time_h3.parquet')
alignment = empirical[['home_gid9', 'qk_empirical']].merge(null, on='home_gid9', validate='one_to_one')
tprox_cells = proximity_groups(
    time[['home_gid9', 'T_prox']]
        .merge(alignment, on='home_gid9', validate='one_to_one')
        .dropna(subset=['qk_empirical', 'qk_null']),
)
time_summary = summarise_proximity(tprox_cells, ['qk_empirical', 'qk_null'])
pooled = time_summary.set_index('T_prox_bin').reindex(TIME_BINS)
minute_x = pooled.x.mask(pooled.index == '40+', 40)

with plt.rc_context({"font.family": "Arial", "font.size": 6, "axes.labelsize": 6}):
    fig, ax = plt.subplots(figsize=(75 / 25.4, 75 / 25.4))
    fig.subplots_adjust(left=.16, right=.97, bottom=.15, top=.95)

    hist = ax.twinx()
    counts, _, _ = hist.hist(
        tprox_cells.T_prox, bins=32, range=(0, 40),
        color="#aaaaaa", alpha=.25, rwidth=1, zorder=0,
    )
    hist.set(yscale="log", ylim=(1, max(float(counts.max()) ** 4, 2)), yticks=[])
    hist.yaxis.set_minor_locator(NullLocator())
    hist.set_axis_off()
    ax.set_zorder(hist.get_zorder() + 1)
    ax.patch.set_visible(False)

    for key, offset in [("null", -.35), ("empirical", .35)]:
        style = SERIES[key]
        metric = f"qk_{key}"
        positions = minute_x + offset
        ax.plot(
            positions, pooled[metric + '_mean'], color=style["colour"], lw=.9,
            ls="-", marker=style["marker"], ms=2.8,
            label=style["label"], clip_on=False, zorder=4,
        )
        q25, q75 = pooled[metric + '_q25'], pooled[metric + '_q75']
        middle = (q25 + q75) / 2
        ax.errorbar(
            positions, middle, yerr=[middle - q25, q75 - middle], fmt="none",
            color=style["colour"], elinewidth=.55, capthick=.55, capsize=2,
            alpha=.55, clip_on=False, zorder=3,
        )

    ax.tick_params(labelsize=5.5)
    ax.grid(False)
    ax.set(xlabel=r"$T_{prox}$ (min)", ylabel=r"$q_K$")
    ax.set_xticks(range(0, 5 * (len(TIME_BINS) + 1), 5), ["0", *TIME_BINS])
    ax.set_xlim(0, 41)
    ax.set_yticks([.30, .40, .50, .60, .70])
    ax.set_ylim(.25, .74)
    ax.legend(frameon=False, fontsize=5.5, loc="upper left")

add_panel_label(ax, "b")
export_figure(fig, "fig3/fig3b-accessibility")

## c–e. Destination attributes

In [ ]:
destination_cities = ["Finland", "Helsinki region", "Oulu", "Tampere", "Turku"]
destination_sets = ("kfreq", "kdist")

# Equal-weighted 1 km grid means and IQRs, matching round-3-analysis.
# Finland uses 6,100 paired grids; 110 grids without null attributes are excluded.
destination_stats = pd.read_parquet(DATA_DIR / 'aggregated' / 'destinations_summary.parquet')
destination_stats = destination_stats.set_index(['city', 'metric', 'scenario', 'location_set']).sort_index()


def draw_destination(ax, city, metric, ylim, yticks):
    positions = np.arange(len(destination_sets))
    for scenario, style in SERIES.items():
        values = destination_stats.loc[(city, metric, scenario)].reindex(destination_sets)
        x = positions + (-.06 if scenario == 'empirical' else .06)
        middle = (values.q25 + values.q75) / 2
        ax.errorbar(
            x, middle, yerr=[middle - values.q25, values.q75 - middle],
            fmt='none', color=style['colour'], elinewidth=.65,
            capthick=.65, capsize=1.6, alpha=.75, zorder=3,
        )
        ax.plot(
            x, values['mean'], color=style['colour'], lw=1.05, ls='-',
            marker=style['marker'], ms=3.4, markeredgecolor='white',
            markeredgewidth=.65, label=style['label'], zorder=4,
        )
    ax.set(title=city.replace(' region', ''), xlim=(-.18, 1.18), ylim=ylim, yticks=yticks)
    ax.set_xticks(positions, [r'$K_{\mathrm{freq}}$', r'$K_{\mathrm{dist}}$'])
    ax.set_axisbelow(True)
    ax.grid(True, axis='both', color='#c9c9c9', linestyle='--', linewidth=.5)


handles = [
    Line2D([0], [0], color=style['colour'], marker=style['marker'], lw=1.05,
           markersize=3.4, markeredgecolor='white', label=style['label'])
    for style in SERIES.values()
]

with plt.rc_context({'axes.labelsize': 7.5, 'xtick.labelsize': 7,
                     'ytick.labelsize': 7, 'legend.fontsize': 7}):
    for label, metric, ylabel, limits, ticks in [
        ('c', 'diversity', 'Div', (.25, .65), [.3, .4, .5, .6]),
        ('d', 'count', 'POI', (0, 100), [0, 25, 50, 75, 100]),
        ('e', 'segregation', 'ExpSeg', (.34, .64), [.35, .4, .45, .5, .55, .6]),
    ]:
        fig, axes = plt.subplots(1, len(destination_cities), sharey=True,
                                 figsize=(172 / 25.4, 56 / 25.4))
        for column, (ax, city) in enumerate(zip(axes, destination_cities)):
            draw_destination(ax, city, metric, limits, ticks)
            ax.set_title(city.replace(' region', ''), fontsize=7.5)
            if column:
                ax.spines['left'].set_visible(False)
                ax.tick_params(axis='y', left=False, labelleft=False)
        axes[0].set_ylabel(ylabel)
        add_panel_label(axes[0], label, x=-.20)
        fig.subplots_adjust(left=.075, right=.99, bottom=.25, top=.83, wspace=.14)
        fig.legend(handles=handles, loc='lower center', ncols=2, frameon=False)
        export_figure(fig, f'fig3/fig3{label}-{metric}')
